# Multiclass Classification Basics: Forest Covertype

A fully offline, reproducible model-making workflow using only `data/covertype.csv`; no code in this notebook makes network requests. The training sample is exactly 350,000 stratified rows. Run this notebook from the project directory after preparing the CSV once if it is absent.

## Seven objectives

(a) Visualize feature relationships.  
(b) Compare Logistic Regression, kNN, and Decision Tree accuracy (with Random Forest as an optional extra).  
(c) Analyze errors using confusion matrices and per-class precision/recall/F1.  
(d) Validate generalization and flag over/underfitting.  
(e) Visualize two-feature decision boundaries.  
(f) Tune hyperparameters with cross-validated search and plot results.  
(g) Recommend a model from the measured test results.

The full training cell below is the same pipeline used by `python train.py`: 70/15/15 stratified split, metrics, five-fold CV, learning/validation curves, tuning, and all registered graphs.

In [ ]:
from pathlib import Path
import sys
import json
import joblib
import pandas as pd
from IPython.display import display, Image, Markdown

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "train.py").exists():
    raise RuntimeError("Run this notebook from the project directory containing train.py")
sys.path.insert(0, str(PROJECT_DIR))


## Step 1 ? Dataset overview and feature relationships

Covertype has 581,012 rows, 54 predictors (10 continuous and 44 binary one-hot columns), and seven imbalanced cover types. The local CSV is sampled to exactly 350,000 rows while preserving class proportions. The summary includes shape, class counts, types, missing values, and descriptive statistics.

In [ ]:
from data import load_and_sample, get_dataset_overview

df = load_and_sample("data/covertype.csv", n_samples=350_000, random_state=42)
overview = get_dataset_overview(df)
print("Sample shape:", overview["shape"], "Source rows:", overview["source_rows"])
print("Missing cells:", overview["missing_values"])
print("Feature types:", {name: len(cols) for name, cols in overview["feature_types"].items()})
display(pd.DataFrame({"count": overview["class_distribution"]["counts"],
                      "percent": overview["class_distribution"]["percentages"]}))
display(overview["summary_statistics"])


### Objective (a) ? EDA plots

Pair plots are class-colored and use a 5,000-row stratified subsample with the top five Random Forest importance features. Correlation heatmaps cover continuous features; box plots compare continuous distributions by class. The class chart makes the imbalance visible.

## Step 2 ? Stratified split, model training, validation, and tuning

The pipeline creates disjoint train/validation/test partitions in 70/15/15 proportions. Logistic Regression and kNN use `StandardScaler` inside their pipelines; Decision Tree and Random Forest do not need scaling. Evaluation subsamples are stratified where required to keep expensive CV, learning-curve, validation-curve, and search runs bounded. kNN and estimators use `n_jobs=-1` where applicable.

In [ ]:
# Runs the complete, reproducible pipeline and saves outputs/ artifacts.
from train import main
main()


### Objective (b), (d), and (f) ? measured comparison and validation

This includes train vs validation accuracy and log loss, five-fold stratified cross-validation (mean ? standard deviation), learning curves, kNN `n_neighbors` and Decision Tree `max_depth` validation curves, and GridSearchCV results. An automatic fit flag uses the train-validation accuracy gap. Final test metrics are computed only after model selection workflows are defined.

In [ ]:
results = joblib.load(PROJECT_DIR / "outputs" / "all_results.joblib")
metrics = pd.DataFrame(results["metrics"]).T
display(metrics)
print(results["recommendation"])


### Graph captions (same registry explanations as the GUI)

Every saved graph is read from the one registry; the caption shown here is the exact one-line explanation used by the Streamlit Graph Gallery.

In [ ]:
registry = json.loads((PROJECT_DIR / "outputs" / "graph_registry.json").read_text(encoding="utf-8"))
for graph in registry:
    path = PROJECT_DIR / graph["file_path"]
    display(Markdown(f"**{graph['title']}**  \n{graph['one_line_explanation']}"))
    display(Image(filename=str(path)))


## Step 3 ? Comparison, confusion matrices, and per-class error analysis

The saved CSV includes accuracy, macro/weighted precision, recall and F1, one-vs-rest macro ROC-AUC, log loss, training time, and test prediction time. Raw and row-normalized confusion matrices are generated for every trained model. Macro metrics give every class equal weight; weighted metrics reflect the observed class frequencies.

In [ ]:
comparison = pd.read_csv(PROJECT_DIR / "outputs" / "metrics_comparison.csv")
display(comparison)
for model_name, report in results["per_class_metrics"].items():
    print("Per-class metrics:", model_name)
    display(pd.DataFrame(report).T)


## Step 4 ? Decision boundaries (Objective e)

Separate two-feature classifiers are trained on a 10,000-row stratified subsample using exactly Elevation and Horizontal_Distance_To_Roadways. The 2-D plot and overlaid observations are pedagogical only: 2 coordinates cannot show all interactions among the original 54 features or fully separate seven overlapping classes.

## Step 5 ? Multiclass theory and live OvR demo

### a) Binary, multiclass, and multilabel

Binary classification chooses one of two labels. Multiclass classification chooses exactly one of three or more mutually exclusive labels (seven here). Multilabel prediction can assign several labels to one example, unlike this dataset.

### b) One-vs-Rest (OvR) and One-vs-One (OvO)

OvR fits K binary classifiers: class k versus all other classes; highest score wins. OvO fits K(K?1)/2 pairwise classifiers and combines their votes (21 classifiers for K=7). Example: if A-vs-rest, B-vs-rest, C-vs-rest scores are 0.75, 0.40, 0.30, predict A. OvR is simple and requires K models, but each problem can be imbalanced and scores need not be jointly calibrated.

In [ ]:
from data import stratified_subsample
from models import build_ovr_pipeline

X_train, X_val, X_test, y_train, y_val, y_test, feature_names = __import__("data").split_data(df)
X_demo, y_demo = stratified_subsample(X_train, y_train, 7000, 42)
ovr_demo_model = build_ovr_pipeline(42).fit(X_demo, y_demo)
print("Base estimator:", ovr_demo_model.named_steps["classifier"].estimator)
print("OvR binary estimators:", len(ovr_demo_model.named_steps["classifier"].estimators_))
print("Predicted label:", ovr_demo_model.predict(X_test[:1])[0])
print("Class probabilities/scores:", ovr_demo_model.predict_proba(X_test[:1])[0])


### c) Softmax / multinomial Logistic Regression

For class k, `P(y=k|x) = exp(w_k?x+b_k) / sum_j exp(w_j?x+b_j)`. The K probabilities sum to one. Training minimizes multiclass cross-entropy, `-sum_i sum_k y_ik log(p_ik)`, with regularization. This project uses scikit-learn LogisticRegression with a multinomial solver setting supplied by its multiclass default.

### d) Native multiclass handling

- **Logistic Regression:** multinomial/softmax probability model in this pipeline; OvR is demonstrated separately.
- **kNN:** directly votes among the K labels of the nearest neighbors; class probabilities are neighbor fractions.
- **Decision Tree:** leaves store class counts/probabilities; splits optimize multiclass impurity.
- **Random Forest:** averages/votes across trees, each of which natively supports K classes.

### e) Model intuition and trade-offs

Logistic Regression learns linear class scores; `C`, solver, and iteration count control regularization and convergence. kNN is a lazy distance-based voter; `n_neighbors`, distance metric, and weighting trade local detail for smoothness. Trees learn threshold rules; `max_depth`, `min_samples_leaf`, and split criterion control complexity. Forests average randomized trees; estimator count, depth, and leaf size affect accuracy, variance, memory, and speed. Scaling matters for Logistic Regression optimization and kNN distances, not tree thresholds.

### f) Splits, cross-validation, overfitting, and curves

The stratified 70/15/15 split reserves validation data for model choices and a test set for final evaluation. Five-fold stratified CV rotates validation folds to estimate score stability. Overfitting appears as a large train-validation gap; underfitting appears when both scores are low. Learning curves vary training-set size; validation curves vary one hyperparameter. These project curves/searches use stratified subsets for practical runtime.

### g) Metrics

- **Accuracy:** correct predictions / all rows; can hide minority-class failures.
- **Precision:** true positives / predicted positives; **recall:** true positives / actual positives.
- **F1:** harmonic mean of precision and recall. **Macro** averages classes equally; **weighted** averages by support.
- **OvR ROC-AUC:** macro average of each class-vs-rest ROC area, using probabilities.
- **Log loss:** negative log probability assigned to the true class; lower is better and confident errors are costly.
- **Confusion matrix:** rows are true classes, columns predictions; diagonal is correct, off-diagonal cells identify confusion.

### h) Decision boundaries

A decision boundary separates predicted regions. Linear Logistic Regression makes planar boundaries; kNN creates local regions; tree models make axis-aligned regions. Here the two specified features alone cannot fully separate seven classes that depend on 54 continuous and binary inputs. The boundary display is a reduced-data illustration, not a substitute for full-feature metrics.

### i) Covertype notes and stratified sample

Forest Covertype represents 30?30 m land patches in Roosevelt National Forest, Colorado. It has 581,012 rows, 54 cartographic predictors (10 continuous, 4 wilderness one-hot, 40 soil one-hot), and seven cover types. Classes 1 and 2 dominate and class 4 is rare. Stratifying the 350,000-row sample preserves this imbalance while making model-making and validation runs more tractable; it does not guarantee identical performance to using every source row.

### j) Recommendation and Iris comparison

The recommendation below is calculated from this run's measured test macro-F1, with test accuracy/log loss and the train-validation gap shown for context. Simple models may work well on Iris because it has only 150 rows, four features, three relatively separable classes, and a balanced class distribution. Covertype has far more data, seven imbalanced classes, mixed 54-dimensional features, and nonlinear class interactions, so those toy-dataset results do not transfer directly.

In [ ]:
print(results["recommendation"])


### k) Jury Q&A

1. **How do these models handle multiclass?** Logistic Regression uses multinomial softmax; kNN, trees, and forests handle K classes natively. OvR is shown for Logistic Regression separately.
2. **Why do simple models suffice on Iris but differ here?** Iris is small, balanced, and relatively separable; Covertype is large, imbalanced, higher-dimensional, and more overlapping.
3. **Why OvR?** It provides a direct K-model extension of a binary estimator; its scores can be less calibrated and subproblems imbalanced.
4. **Why scale for kNN?** Distances otherwise become dominated by larger-scale features. Scaling also helps Logistic Regression optimization.
5. **Why can trees overfit?** Deep trees can memorize training rows; depth and leaf constraints regularize them.
6. **Why log loss?** It evaluates probability quality and penalizes confident incorrect predictions.
7. **Why macro-F1?** It weights each class equally, exposing minority-class failure hidden by weighted scores.
8. **How assess overfitting?** Train-validation gap, learning curves, CV variability, and final held-out test results provide complementary evidence.
9. **Why 350k rows?** It is a large stratified sample that preserves class mix while bounding model and experiment cost.
10. **What are limitations?** One fixed split and bounded searches, class imbalance remains, only baseline models are compared, and two-feature boundaries omit 52 inputs; local results may not generalize to other forests.

## Final reproducibility notes

The CSV is the only input. Generated model files, comparison CSV, registry, and graph images live under `outputs/`. This notebook and the Streamlit GUI share the same saved result artifacts and exact registered graph explanations.